# 05 — Qwen2.5-VL on RHpE: open-weight MLLM replication of the Gemini Lesson 14 protocol

**Branch:** `experiment/qwen-mlx`
**Spec:** [`docs/qwen-experiment-spec.md`](../docs/qwen-experiment-spec.md)
**Run date:** 2026-05-07 (M2 Max, 96 GB, mlx-vlm 0.5.0 + mlx 0.31.2)
**Inference backend:** MLX (Apple Silicon native — no PyTorch MPS fallback needed)
**Model:** `mlx-community/Qwen2.5-VL-7B-Instruct-bf16` (closest match to spec's "fp16 ~14 GB"; the unsuffixed `Qwen2.5-VL-7B-Instruct` doesn't exist in the mlx-community org)

**Goal.** Replicate the three-probe Gemini Lesson 14 protocol on an open-weight, self-hosted multimodal LLM with a different post-training regime. Compare against the existing Gemini 2.5 Pro and Gemini 3.1 Pro Preview JSONLs in `outputs/`. Pre-registered §4 outcome interpretation in the spec.

**Headline.** All three failure modes reproduced (refusal-bias collapse, cross-rep instability, perception/classification decoupling). §4 row 1 hit. 32B follow-up gated and skipped per spec. See [Lesson 15](../docs/lessons_learned.md) and [`outputs/qwen_vs_gemini_comparison.md`](../outputs/qwen_vs_gemini_comparison.md).


## 1. Setup

In [ ]:
import os, sys, json
from pathlib import Path

POC = Path(os.getcwd()).resolve()
if POC.name == 'notebooks':
    POC = POC.parent
sys.path.insert(0, str(POC / 'tools'))
print('POC:', POC)
print('outputs/ exists:', (POC / 'outputs').exists())

try:
    import mlx_vlm
    print('mlx-vlm:', mlx_vlm.__version__ if hasattr(mlx_vlm, '__version__') else 'imported')
    import mlx.core as mx
    print('mlx core device:', mx.default_device())
except ImportError:
    print('mlx-vlm not installed — Apple Silicon required. Run `bash setup.sh` on Darwin/arm64.')
    print('Skip the rest of this notebook on Linux/Colab.')


## 2. Smoke test: one clip with prompt A

Re-runs a single inference against `Qwen2.5-VL-7B-Instruct-bf16` using prompt A from `tools/gemini_audit.py` (verbatim). Confirms the model loads, the chat template handles `{type: video, fps: 10}` content, and the output parser strips the markdown JSON fence cleanly.

First execution downloads ~14 GB. Subsequent runs reuse the HF cache and load in ~5 s.


In [ ]:
import time
import gemini_audit as ga
import qwen_audit as qa

rows = qa.load_36_clip_set()
clip0 = rows[0]
print(f'Smoke clip: {clip0["video_rel"]} (truth: {clip0["true_label"]})')

runner = qa.QwenRunner(model_size='7B')
t0 = time.time()
result = runner.run(clip0['clip'], ga.PROMPT_A, temperature=0.0)
print(f'Inference: {time.time()-t0:.1f}s')
print(f'prompt_tokens={result["prompt_tokens"]}, generation_tokens={result["candidate_tokens"]}')
print()
print(result['text'])
parsed = qa.parse_qwen_json(result['text'])
print()
print(f'Parsed: classification={parsed.get("classification")}  confidence={parsed.get("confidence")}')


## 3. Run the three probes

The full protocol lives in `tools/qwen_audit.py`. All probes are idempotent (skip clips already in JSONL) and append-only. From the project root:

```bash
.venv/bin/python tools/qwen_audit.py --probe 1   # promptA, t=0
.venv/bin/python tools/qwen_audit.py --probe 2   # promptC + sys, Qwen defaults
.venv/bin/python tools/qwen_audit.py --probe 3   # description, 5 reps × 10 clips, t=0.7
```

Total ≈ 10 min on M2 Max, $0.


## 4. Analysis: agreement + bg-rate, side-by-side with Gemini

In [ ]:
import json
from collections import defaultdict

def load(p):
    p = Path(p)
    return [json.loads(l) for l in open(p)] if p.exists() else []

OUTPUTS = POC / 'outputs'
configs = [
    ('Gemini 2.5 Pro · A',  'gemini_audit_results_gemini-2.5-pro_promptA.jsonl',         'gemini_label'),
    ('Gemini 2.5 Pro · B',  'gemini_audit_results_gemini-2.5-pro_promptB.jsonl',         'gemini_label'),
    ('Gemini 3.1 Pre · A',  'gemini_audit_results_gemini-3.1-pro-preview_promptA.jsonl', 'gemini_label'),
    ('Gemini 3.1 Pre · B',  'gemini_audit_results_gemini-3.1-pro-preview_promptB.jsonl', 'gemini_label'),
    ('Gemini 3.1 Pre · C',  'gemini_audit_results_gemini-3.1-pro-preview_promptC.jsonl', 'gemini_label'),
    ('Qwen 7B-bf16  · A',   'qwen25vl_7b_promptA.jsonl',                                 'qwen_label'),
    ('Qwen 7B-bf16  · C',   'qwen25vl_7b_promptC.jsonl',                                 'qwen_label'),
]

print(f'{"config":22} {"n":>3} {"err":>3} {"agree":>14} {"bg-rate":>12}')
print('-' * 60)
for name, path, lf in configs:
    rows = load(OUTPUTS / path)
    if not rows:
        print(f'{name:22} (file missing)'); continue
    err = sum(r.get('error') is not None for r in rows)
    rok = [r for r in rows if r.get('error') is None and r.get(lf) is not None]
    agree = sum(r[lf] == r['true_label'] for r in rok)
    bg = sum(r[lf] == 'background' for r in rok)
    n = len(rok)
    print(f'{name:22} {len(rows):3d} {err:3d} {f"{agree}/{n}={agree/n:.3f}":>14} {f"{bg}/{n}={bg/n:.3f}":>12}')


## 5. Cross-rep stability + decoupling (description probe)

In [ ]:
import re

MOTION = re.compile(r'\b(twitch|rotat|swivel|flick|moving|pinning|swaying|displacement|active|change|adjust(?:s|ing)?|shift|tilt)\b', re.I)
HEDGE  = re.compile(r'\b(minimal|slight(?:ly)?|barely|hardly|small|subtle|occasional(?:ly)?|mostly|relatively|rarely|infrequent|negligible)\b', re.I)
STILL  = re.compile(r'\b(still|stationary|relaxed|upright|fixed|steady|stable|consistent|remain(?:s|ed|ing)?|holding)\b', re.I)

def cls(text):
    if not text: return 'still'
    m, h, s = bool(MOTION.search(text)), bool(HEDGE.search(text)), bool(STILL.search(text))
    if m and not h: return 'motion'
    if m and h: return 'mixed'
    if s or h: return 'still'
    return 'still'

def probe(label, fname, field='observation'):
    rows = load(OUTPUTS / fname)
    if not rows:
        print(f'  {label}: missing'); return
    by = defaultdict(list)
    for r in rows:
        by[r['clip']].append(cls(r.get(field) or r.get('reasoning') or ''))
    n = len(by)
    stable = sum(1 for v in by.values() if len(set(v)) == 1)
    mm = sum(1 for v in by.values() if (v.count('motion') + v.count('mixed')) >= 3)
    print(f'  {label:42}: {n} clips, stable {stable}/{n}, motion-majority {mm}/{n}')

print('Cross-rep stability (5/5 same label) + motion-majority on description-only probe:')
probe('Gemini 2.5 Pro · t=1.0',                'gemini_audit_probe_gemini-2.5-pro_temp1.0.jsonl')
probe('Gemini 3.1 Pre · t1.0 thinkLOW (10)',   'gemini_audit_probe_gemini-3.1-pro-preview_temp1.0_thinkLOW.jsonl')
probe('Gemini 3.1 Pre · t1.0 thinkLOW (N=20)', 'gemini_audit_probe_gemini-3.1-pro-preview_temp1.0_thinkLOW_N20.jsonl')
probe('Qwen 7B-bf16   · t=0.7',                'qwen25vl_7b_description_probe.jsonl')


## 6. §4 outcome and decision

| Spec §4 row | Threshold | Qwen 7B observed | Hit? |
|---|---|---|---|
| **Row 1 — all 3 failure modes reproduced** | refusal-bias + cross-rep instability + decoupling all present | bg-rate 100 %, 0/10 stable, 3/10 decoupling | **YES** |
| Row 2 — materially cleaner | agreement ≥ 0.70 AND bg-rate < 80 % AND no > 3 pp decoupling | 0.611, 100 %, 30 % | NO |
| Row 3 — mixed | partial improvement | none clean | NO |
| Row 4 — infrastructure failure | wouldn't run | 0/122 errors | NO |

**Decision: Row 1.** No 32B follow-up. Lesson 14 generalizes from "Gemini family" to "MLLM class" within the scope tested. The MLLM-as-classifier track is closed for this task.

V-JEPA-2 + linear probe (LOSO 0.875) remains the spine. See [Lesson 15](../docs/lessons_learned.md) for the full writeup and [`outputs/qwen_vs_gemini_comparison.md`](../outputs/qwen_vs_gemini_comparison.md) for the side-by-side numbers.
